# Notebook 13 — Analyse détaillée des échecs de localisation

## Contexte

La localisation actuelle atteint Top-1 = 11.9% avec les traces seules.
Ce notebook explore les échecs pour identifier des patterns exploitables
et créer des signatures spécialisées par type de panne.

## Objectifs

1. Comprendre pourquoi ts-contacts (panne return) n'apparaît pas dans le top-5
2. Identifier les logs Java exploitables pour la panne exception
3. Analyser les patterns de latence pour network_delay
4. Proposer des améliorations concrètes

In [1]:
import pickle
import csv
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from datetime import timedelta
from collections import Counter, defaultdict
from sklearn.neighbors import LocalOutlierFactor
from sklearn.ensemble import IsolationForest
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

PROJET    = Path('/home/eunice/Bureau/Train_ticket/Intelligent_observability')
NORMAL    = PROJET / 'data/normal'
ANOMALIES = PROJET / 'data/anomalies'
MODELS_DIR = PROJET / 'models'
OUTPUT     = PROJET / 'output'
RESULTS    = PROJET / 'results'
FIGURES    = PROJET / 'figures/localisation_avancee'
FIGURES.mkdir(parents=True, exist_ok=True)

DATES_TT = ['2023-01-29', '2023-01-30']
FEATURES_SPAN = ['duration_ms']

# ─── Charger le ground truth ───
gt = pd.read_csv(OUTPUT / 'ground_truth.csv')
print(f"Ground truth : {len(gt)} fenêtres")
print(f"Types de pannes : {sorted(gt['fault_type'].unique())}")
print(f"\nRépartition :")
print(gt['fault_type'].value_counts())

# ─── Charger les modèles pré-entraînés ───
with open(MODELS_DIR / 'lof_tt.pkl', 'rb') as f:
    lof_data = pickle.load(f)
    modeles_lof = lof_data['modeles']
    scalers_lof = lof_data['scalers']

with open(MODELS_DIR / 'tfidf_tt.pkl', 'rb') as f:
    tfidf_data = pickle.load(f)
    tfidf = tfidf_data['vectorizer']
    vecteur_ref = tfidf_data['vecteur_ref']

with open(MODELS_DIR / 'if_traces_tt.pkl', 'rb') as f:
    if_data = pickle.load(f)
    modeles_if = if_data['modeles']
    scalers_if = if_data['scalers']

print(f"\n✓ Modèles chargés :")
print(f"  LOF     : {len(modeles_lof)} services")
print(f"  TF-IDF  : {len(tfidf.vocabulary_)} termes")
print(f"  IF      : {len(modeles_if)} services")

# ─── Fonctions de chargement ───
def charger_logs(date, source, fenetre):
    chemin = source / date / 'log' / f'{fenetre}_log.csv'
    if not chemin.exists():
        return pd.DataFrame()
    colonnes = ['Timestamp','TimeUnixNano','Node','PodName',
                'Container','TraceID','SpanID','Log']
    rows = []
    with open(chemin, 'r', encoding='utf-8', errors='replace') as f:
        reader = csv.reader(f)
        next(reader)
        for row in reader:
            if len(row) >= 8:
                rows.append(row[:8])
    if not rows:
        return pd.DataFrame()
    df = pd.DataFrame(rows, columns=colonnes)
    df['service'] = df['PodName'].apply(lambda x: str(x).rsplit('-', 2)[0])
    return df

def charger_traces(date, source, fenetre):
    chemin = source / date / 'trace' / f'{fenetre}_trace.csv'
    if not chemin.exists():
        return pd.DataFrame()
    df = pd.read_csv(chemin, on_bad_lines='skip')
    df['duration_ms'] = pd.to_numeric(df['Duration'], errors='coerce') / 1e6
    df['service'] = df['PodName'].apply(lambda x: str(x).rsplit('-', 2)[0])
    return df

print("\n✓ Fonctions prêtes")

Ground truth : 135 fenêtres
Types de pannes : ['cpu_contention', 'exception', 'network_delay', 'return']

Répartition :
fault_type
network_delay     42
exception         39
return            33
cpu_contention    21
Name: count, dtype: int64

✓ Modèles chargés :
  LOF     : 46 services
  TF-IDF  : 355 termes
  IF      : 28 services

✓ Fonctions prêtes


## 2. Analyse par type de panne — quelles fenêtres échouent ?

On mesure pour chaque type de panne :
- Où se trouve le vrai coupable dans le ranking actuel
- Quels services sont fréquemment détectés à tort en top-1
- Y a-t-il des patterns exploitables pour améliorer

In [2]:
# 
# LOCALISATION ACTUELLE (traces + fusion RRF)
# 

def localiser_traces(df_fen):
    """Ranking par IF sur spans (méthode actuelle)."""
    scores = {}
    for service in df_fen['service'].unique():
        if service not in modeles_if:
            continue
        df_svc = df_fen[df_fen['service'] == service][FEATURES_SPAN].dropna()
        if df_svc.empty:
            continue
        X = scalers_if[service].transform(df_svc)
        pred = modeles_if[service].predict(X)
        scores[service] = (pred == -1).sum() / len(pred)
    return sorted(scores.items(), key=lambda x: x[1], reverse=True)

# 
# ANALYSE PAR TYPE DE PANNE
# 
print("=== Analyse détaillée par type de panne ===\n")

analyses = defaultdict(list)

for _, row in gt.iterrows():
    df_traces = charger_traces(row['date'], ANOMALIES, row['window'])
    if df_traces.empty:
        continue
    
    ranking = localiser_traces(df_traces)
    services = [s for s, _ in ranking]
    vrai = row['faulty_service']
    
    if vrai in services:
        rang = services.index(vrai) + 1
    else:
        rang = -1  # non trouvé
    
    analyses[row['fault_type']].append({
        'window'    : row['window'],
        'vrai'      : vrai,
        'rang_vrai' : rang,
        'top1_predit': services[0] if services else None,
        'ranking'   : services[:5],
    })

# 
# STATISTIQUES PAR TYPE
# 
for fault_type, cas in analyses.items():
    n = len(cas)
    rangs_trouves = [c['rang_vrai'] for c in cas if c['rang_vrai'] > 0]
    non_trouves = sum(1 for c in cas if c['rang_vrai'] == -1)
    
    print(f"\n{'='*60}")
    print(f"Type : {fault_type}  ({n} fenêtres)")
    print(f"{'='*60}")
    
    if rangs_trouves:
        print(f"  Position moyenne du vrai coupable : {np.mean(rangs_trouves):.1f}")
        print(f"  Position médiane                  : {np.median(rangs_trouves):.0f}")
        print(f"  Top-1 (rang 1)                    : {sum(1 for r in rangs_trouves if r==1)}/{n}")
        print(f"  Top-3 (rang ≤ 3)                  : {sum(1 for r in rangs_trouves if r<=3)}/{n}")
        print(f"  Top-10 (rang ≤ 10)                : {sum(1 for r in rangs_trouves if r<=10)}/{n}")
    print(f"  Non trouvé dans le ranking        : {non_trouves}/{n}")
    
    # Services top-1 les plus fréquents (à tort)
    predits_top1 = [c['top1_predit'] for c in cas if c['top1_predit'] and c['top1_predit'] != c['vrai']]
    top_faux = Counter(predits_top1).most_common(5)
    if top_faux:
        print(f"\n  Services les plus souvent prédits à tort en top-1 :")
        for s, count in top_faux:
            print(f"    {s:<40} : {count} fois")

# 
# CONCLUSION
# 
print(f"\n\n{'='*60}")
print("CONCLUSION DE L'ANALYSE")
print(f"{'='*60}")

# Diagnostic
total_top10 = sum(
    sum(1 for c in cas if c['rang_vrai'] > 0 and c['rang_vrai'] <= 10)
    for cas in analyses.values()
)
print(f"\n  Vrai coupable dans top-10 : {total_top10}/135 = {total_top10/135*100:.1f}%")
print(f"  → Si on améliore le RE-RANKING du top-10, potentiel gros gain")

=== Analyse détaillée par type de panne ===


Type : return  (33 fenêtres)
  Position moyenne du vrai coupable : 12.6
  Position médiane                  : 15
  Top-1 (rang 1)                    : 3/33
  Top-3 (rang ≤ 3)                  : 8/33
  Top-10 (rang ≤ 10)                : 13/33
  Non trouvé dans le ranking        : 2/33

  Services les plus souvent prédits à tort en top-1 :
    ts-food-service                          : 20 fois
    ts-cancel-service                        : 4 fois
    ts-execute-service                       : 2 fois
    ts-preserve-service                      : 1 fois
    ts-delivery-service                      : 1 fois

Type : exception  (39 fenêtres)
  Position moyenne du vrai coupable : 14.3
  Position médiane                  : 13
  Top-1 (rang 1)                    : 0/39
  Top-3 (rang ≤ 3)                  : 0/39
  Top-10 (rang ≤ 10)                : 8/39
  Non trouvé dans le ranking        : 0/39

  Services les plus souvent prédits à tort en top-1 

In [3]:
# 
# DIAGNOSTIC — ts-food-service
# 
print("=== Diagnostic ts-food-service ===\n")

# 1. Combien de spans normaux avons-nous pour ce service ?
df_normal_traces = pd.DataFrame()
for date in DATES_TT:
    for f in (NORMAL / date / 'trace').glob('*.csv'):
        df = charger_traces(date, NORMAL, f.stem.replace('_trace', ''))
        df_normal_traces = pd.concat([df_normal_traces, df], ignore_index=True)

if not df_normal_traces.empty:
    df_food_normal = df_normal_traces[df_normal_traces['service'] == 'ts-food-service']
    print(f"Spans normaux ts-food-service : {len(df_food_normal)}")
    if not df_food_normal.empty:
        print(f"  Durée moyenne  : {df_food_normal['duration_ms'].mean():.2f} ms")
        print(f"  Durée médiane  : {df_food_normal['duration_ms'].median():.2f} ms")
        print(f"  Durée max      : {df_food_normal['duration_ms'].max():.2f} ms")

# 2. Est-il vraiment "toujours anormal" ?
print(f"\n=== Taux d'anomalies par service (moyenne sur 135 fenêtres) ===\n")
taux_par_service = defaultdict(list)

for _, row in gt.iterrows():
    df_fen = charger_traces(row['date'], ANOMALIES, row['window'])
    if df_fen.empty:
        continue
    for service in df_fen['service'].unique():
        if service not in modeles_if:
            continue
        df_svc = df_fen[df_fen['service'] == service][FEATURES_SPAN].dropna()
        if df_svc.empty:
            continue
        X = scalers_if[service].transform(df_svc)
        pred = modeles_if[service].predict(X)
        taux = (pred == -1).sum() / len(pred)
        taux_par_service[service].append(taux)

# Afficher les services avec le plus haut taux moyen
print("Top 10 services les plus souvent 'anormaux' :")
print(f"{'Service':<40} {'Taux moyen':>12} {'Nb fenêtres':>15}")
print("-" * 70)
resume = [(s, np.mean(t), len(t)) for s, t in taux_par_service.items()]
for s, taux, n in sorted(resume, key=lambda x: x[1], reverse=True)[:10]:
    marker = "  ← parasite ?" if taux > 0.5 else ""
    print(f"{s:<40} {taux:>11.2%} {n:>15}{marker}")

=== Diagnostic ts-food-service ===

Spans normaux ts-food-service : 120
  Durée moyenne  : 0.02 ms
  Durée médiane  : 0.01 ms
  Durée max      : 0.12 ms

=== Taux d'anomalies par service (moyenne sur 135 fenêtres) ===

Top 10 services les plus souvent 'anormaux' :
Service                                    Taux moyen     Nb fenêtres
----------------------------------------------------------------------
ts-food-service                               65.43%             120  ← parasite ?
ts-cancel-service                             39.99%             107
ts-verification-code-service                  36.89%             134
ts-order-service                              33.01%             135
ts-security-service                           32.23%             131
ts-preserve-service                           29.31%             124
ts-gateway-service                            29.25%             135
ts-train-service                              28.83%             133
ts-order-other-service      

In [4]:
# ═══════════════════════════════════════════
# LOCALISATION AMÉLIORÉE V2 — score relatif au baseline
# ═══════════════════════════════════════════
print("=== Amélioration V2 — score relatif au baseline ===\n")

# 1. Calculer le taux "normal" (moyenne) pour chaque service
baseline_taux = {s: np.mean(t) for s, t in taux_par_service.items()}

# 2. Nouvelle fonction de localisation
def localiser_traces_v2(df_fen):
    """Ranking basé sur (taux_observé - baseline_service)."""
    scores = {}
    for service in df_fen['service'].unique():
        if service not in modeles_if:
            continue
        df_svc = df_fen[df_fen['service'] == service][FEATURES_SPAN].dropna()
        if df_svc.empty:
            continue
        X = scalers_if[service].transform(df_svc)
        pred = modeles_if[service].predict(X)
        taux_observe = (pred == -1).sum() / len(pred)
        # Score = déviation par rapport au taux moyen du service
        baseline = baseline_taux.get(service, 0)
        scores[service] = taux_observe - baseline
    return sorted(scores.items(), key=lambda x: x[1], reverse=True)

# 3. Évaluer V2
print("Évaluation V2 sur les 135 fenêtres...")
top1_v2 = top3_v2 = top5_v2 = top10_v2 = 0
rangs_v2 = []

for _, row in gt.iterrows():
    df_fen = charger_traces(row['date'], ANOMALIES, row['window'])
    if df_fen.empty:
        continue
    ranking = localiser_traces_v2(df_fen)
    services = [s for s, _ in ranking]
    vrai = row['faulty_service']
    if vrai in services:
        rang = services.index(vrai) + 1
        rangs_v2.append(rang)
        if rang == 1: top1_v2 += 1
        if rang <= 3: top3_v2 += 1
        if rang <= 5: top5_v2 += 1
        if rang <= 10: top10_v2 += 1

n = len(gt)
mrr_v2 = sum(1/r for r in rangs_v2) / n if rangs_v2 else 0

print(f"\n=== Résultats V2 (traces avec baseline correction) ===")
print(f"  Top-1  : {top1_v2}/{n} = {top1_v2/n*100:.1f}% (vs 11.9% avant)")
print(f"  Top-3  : {top3_v2}/{n} = {top3_v2/n*100:.1f}%")
print(f"  Top-5  : {top5_v2}/{n} = {top5_v2/n*100:.1f}%")
print(f"  Top-10 : {top10_v2}/{n} = {top10_v2/n*100:.1f}%")
print(f"  MRR    : {mrr_v2:.3f}")

# Par type de panne
print(f"\n=== Top-1 par type de panne (V2) ===")
for ft in sorted(gt['fault_type'].unique()):
    df_ft = gt[gt['fault_type'] == ft]
    corrects = 0
    for _, row in df_ft.iterrows():
        df_fen = charger_traces(row['date'], ANOMALIES, row['window'])
        if df_fen.empty:
            continue
        ranking = localiser_traces_v2(df_fen)
        if ranking and ranking[0][0] == row['faulty_service']:
            corrects += 1
    print(f"  {ft:<20} : {corrects}/{len(df_ft)} = {corrects/len(df_ft)*100:.1f}%")

=== Amélioration V2 — score relatif au baseline ===

Évaluation V2 sur les 135 fenêtres...

=== Résultats V2 (traces avec baseline correction) ===
  Top-1  : 12/135 = 8.9% (vs 11.9% avant)
  Top-3  : 20/135 = 14.8%
  Top-5  : 28/135 = 20.7%
  Top-10 : 62/135 = 45.9%
  MRR    : 0.190

=== Top-1 par type de panne (V2) ===
  cpu_contention       : 11/21 = 52.4%
  exception            : 0/39 = 0.0%
  network_delay        : 0/42 = 0.0%
  return               : 1/33 = 3.0%


In [ ]:
# 
# SIGNATURE EXCEPTION — analyse des logs
# 
print("=== Signature Exception dans les logs ===\n")

# Mots-clés typiques des exceptions Java Spring Boot
MOTS_EXCEPTION = [
    'Exception', 'Error', 'ERROR', 'Caused by',
    'at java.', 'at org.', 'at com.',
    'stack trace', 'stacktrace',
    'NullPointerException', 'RuntimeException',
    'IllegalArgumentException', 'ClassCastException',
    'IndexOutOfBoundsException', 'NumberFormatException',
    'IOException', 'SQLException',
    'throw', 'threw', 'thrown',
]

def scorer_service_exception(df_logs, service):
    """
    Score un service par nombre de mots-clés d'exception dans ses logs.
    """
    df_svc = df_logs[df_logs['service'] == service]
    if df_svc.empty:
        return 0
    
    texte = ' '.join(df_svc['Log'].astype(str).tolist())
    score = 0
    for mot in MOTS_EXCEPTION:
        score += texte.count(mot)
    
    return score

def localiser_par_signature_exception(df_logs):
    """Ranking par signature de mots-clés d'exception."""
    scores = {}
    for service in df_logs['service'].unique():
        score = scorer_service_exception(df_logs, service)
        if score > 0:
            scores[service] = score
    return sorted(scores.items(), key=lambda x: x[1], reverse=True)

# 
# ÉVALUATION SUR LES 39 FENÊTRES EXCEPTION
# 
print("Test sur les 39 fenêtres de type 'exception'...\n")

df_exception = gt[gt['fault_type'] == 'exception']
top1 = top3 = top5 = 0
rangs = []
non_trouves = 0

for _, row in df_exception.iterrows():
    df_logs = charger_logs(row['date'], ANOMALIES, row['window'])
    if df_logs.empty:
        non_trouves += 1
        continue
    
    ranking = localiser_par_signature_exception(df_logs)
    services = [s for s, _ in ranking]
    vrai = row['faulty_service']
    
    if vrai in services:
        rang = services.index(vrai) + 1
        rangs.append(rang)
        if rang == 1: top1 += 1
        if rang <= 3: top3 += 1
        if rang <= 5: top5 += 1
    else:
        non_trouves += 1

n = len(df_exception)
mrr = sum(1/r for r in rangs) / n if rangs else 0

print(f"=== Résultats — Signature Exception ===")
print(f"  Top-1  : {top1}/{n} = {top1/n*100:.1f}% (vs 0% traces seules)")
print(f"  Top-3  : {top3}/{n} = {top3/n*100:.1f}%")
print(f"  Top-5  : {top5}/{n} = {top5/n*100:.1f}%")
print(f"  Non trouvés : {non_trouves}/{n}")
print(f"  MRR    : {mrr:.3f}")

# Diagnostic — quels services ressortent le plus souvent
print(f"\n=== Services les plus souvent en top-1 (vrais et faux) ===")
top1_predits = []
for _, row in df_exception.iterrows():
    df_logs = charger_logs(row['date'], ANOMALIES, row['window'])
    if df_logs.empty:
        continue
    ranking = localiser_par_signature_exception(df_logs)
    if ranking:
        top1_predits.append({
            'predit': ranking[0][0],
            'vrai'  : row['faulty_service'],
            'score' : ranking[0][1],
            'correct': ranking[0][0] == row['faulty_service'],
        })

top_predits_frequency = Counter([p['predit'] for p in top1_predits])
print(f"\n  Top 5 services prédits en top-1 :")
for s, count in top_predits_frequency.most_common(5):
    corrects_pour_s = sum(1 for p in top1_predits if p['predit'] == s and p['correct'])
    print(f"    {s:<40} : {count} fois (correct: {corrects_pour_s})")

=== Signature Exception dans les logs ===

Test sur les 39 fenêtres de type 'exception'...

=== Résultats — Signature Exception ===
  Top-1  : 2/39 = 5.1% (vs 0% traces seules)
  Top-3  : 6/39 = 15.4%
  Top-5  : 6/39 = 15.4%
  Non trouvés : 33/39
  MRR    : 0.103

=== Services les plus souvent en top-1 (vrais et faux) ===

  Top 5 services prédits en top-1 :
    ts-food-service                          : 10 fois (correct: 0)
    ts-travel-service                        : 9 fois (correct: 2)
    ts-preserve-other-service                : 8 fois (correct: 0)
    ts-travel2-service                       : 6 fois (correct: 0)
    ts-preserve-service                      : 4 fois (correct: 0)


In [6]:
# Prendre une fenêtre exception où on a échoué
row_test = df_exception.iloc[0]
print(f"Fenêtre : {row_test['date']} {row_test['window']}")
print(f"Vrai coupable : {row_test['faulty_service']}")

df_logs = charger_logs(row_test['date'], ANOMALIES, row_test['window'])

# Combien de logs par service ?
print(f"\nNombre de logs par service dans cette fenêtre :")
for s, n in df_logs['service'].value_counts().head(10).items():
    marker = " ← VRAI COUPABLE" if s == row_test['faulty_service'] else ""
    print(f"  {s:<40} : {n}{marker}")

# Voir les logs du vrai coupable
df_vrai = df_logs[df_logs['service'] == row_test['faulty_service']]
print(f"\n=== Extraits de logs du vrai coupable ({len(df_vrai)} logs) ===\n")
if not df_vrai.empty:
    for i, log in enumerate(df_vrai['Log'].head(10)):
        log_str = str(log)[:200]
        print(f"{i+1}. {log_str}")

# Chercher des patterns anormaux
print(f"\n=== Mots-clés trouvés dans ces logs ===")
tous_logs_vrai = ' '.join(df_vrai['Log'].astype(str).tolist())
for mot in MOTS_EXCEPTION:
    count = tous_logs_vrai.count(mot)
    if count > 0:
        print(f"  '{mot}' : {count} fois")

Fenêtre : 2023-01-29 09_25
Vrai coupable : ts-basic-service

Nombre de logs par service dans cette fenêtre :
  ts-seat-service                          : 350
  ts-basic-service                         : 336 ← VRAI COUPABLE
  ts-travel-service                        : 198
  ts-order-service                         : 142
  ts-config-service                        : 132
  ts-travel2-service                       : 110
  ts-order-other-service                   : 107
  ts-preserve-service                      : 69
  ts-cancel-service                        : 67
  ts-station-service                       : 48

=== Extraits de logs du vrai coupable (336 logs) ===

1. {"log":"17:24:43.293 INFO  f.m.s.BasicServiceImpl#119 TraceID: eb85592b1d4dc5aa90b70824916e7880 SpanID: be74ee68d175ff7a [queryForTravel][all done][result: TravelResult(status=true, percent=1.0, train
2. {"log":"17:24:43.293 INFO  f.m.s.BasicServiceImpl#100 TraceID: eb85592b1d4dc5aa90b70824916e7880 SpanID: be74ee68d175ff7a [quer